Open h5 results file

In [6]:
# Open h5 file and import necessary modules
import h5py

def inspect_h5_file(file_path):
    def print_structure(name, obj):
        # Evaluate needed identation based on the depth in the hierarchy
        indent = '  ' * name.count('/')

        if isinstance(obj, h5py.Group):
            print(f"{indent}Group: {name}")
        elif isinstance(obj, h5py.Group):
            print(f"{indent}Dataset: {name} - Shape: {obj.shape}, Dtype: {obj.dtype}")

            if obj.attrs:
                for attr_name, attr_value in obj.attrs.items():
                    print(f"{indent}  Attribute: {attr_name} = {attr_value}")

    print(f"Inspecting HDF5 file: {file_path}")
    with h5py.File(file_path, 'r') as h5file:
        h5file.visititems(print_structure)

# Example usage
file_path = r'output\20260205175237-1\optimization_results.h5'
inspect_h5_file(file_path)

Inspecting HDF5 file: output\20260205175237-1\optimization_results.h5
Group: design
  Group: design/networks
    Group: design/networks/2022
      Group: design/networks/2022/hydrogenPipelineOnshore
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL1BEL2
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL1DE
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL1NL1
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL1NL2
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL1NL3
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL2BEL1
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL2DE
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL2NL1
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL2NL2
        Group: design/networks/2022/hydrogenPipelineOnshore/BEL2NL3
        Group: design/networks/2022/hydrogenPipelineOnshore/DEBEL1
        Group: design/networks/2022/hydrogenPipelineOn

In [5]:
from __future__ import annotations
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

def _to_scalar(x):
    """
    Convert dataset HDF5 to Python scalar if possible.
    Handles: scalars, 0-d arrays, 1-element arrays.
    """
    if isinstance(x, (np.generic,)):
        return float(x.item())
    x = np.asarray(x)
    if x.shape == ():          # 0-d
        return float(x.item())
    if x.size == 1:
        return float(x.reshape(-1)[0].item())
    return x  # if it's a real array (should not be for capex_tot)

def create_base_df(h5_path: str | Path,
                   year: str = "2022") -> pd.DataFrame:
    """
    Check structure of the HDF5 file and create a base DataFrame for design results.
    DF structure:
    | node | component |
    """
    h5_path = Path(h5_path)

    rows = []
    base = f"design/nodes/{year}"
    
    with h5py.File(h5_path, "r") as f:
        if base not in f:
            raise KeyError(f"Path '{base}' not found in the file.")

        g_year = f[base]

        for node in g_year.keys():
            g_node_path = f"{base}/{node}"
            g_node = f[g_node_path]

            for comp in g_node.keys():
                rows.append({"node": node, "component": comp})
        
    df = pd.DataFrame(rows)

    return df
        

def load_design_results(dataset_name: str,
                      h5_path: str | Path,
                      year: str = "2022",
                      base_df: pd.DataFrame | None = None) -> pd.DataFrame:
    """
    Extracts dataset_name for each component: design/nodes/{year}/{node}/{component}/{dataset_name}
    Adds dataset_name to the base DataFrame created by create_base_df.
    DF structure:
    | node | component | dataset_name |
    """
    h5_path = Path(h5_path)
    base = f"design/nodes/{year}"

    if base_df is None:
        df = create_base_df(h5_path, year)
    else:
        df = base_df.copy()
        if not isinstance(df.index, pd.MultiIndex) or df.index.names != ['node', 'component']:
            df = df.set_index(['node', 'component']).sort_index()

    # Add dataset_name to the base DataFrame created by create_base_df
    df[dataset_name] = np.nan  # initialize column dataset_name with NaN

    with h5py.File(h5_path, "r") as f:
        if base not in f:
            raise KeyError(f"Path '{base}' not found in the file.")
        
        g_year = f[base]

        for node in g_year.keys():
            g_node_path = f"{base}/{node}"
            g_node = f[g_node_path]

            for comp in g_node.keys():
                ds_path = f"{g_node_path}/{comp}/{dataset_name}"
                if ds_path in f:
                    val = _to_scalar(f[ds_path][()])
                    # # Check if the node/component pair exists in the base DataFrame
                    # mask = (df['node'] == node) & (df['component'] == comp)
                    # if mask.any():
                    #     df.loc[mask, dataset_name] = val
                    # else:
                    #     print(f"Warning: node/component pair ({node}, {comp}) not found in base DataFrame. Skipping {dataset_name} for this pair.")
                    df.loc[(node, comp), dataset_name] = val

    return df.reset_index()

results_path = r"output\20260206153536-1\optimization_results.h5"
year = '2022'

# Create standard df for design results
components_df = create_base_df(results_path, year)

# Load capex data
components_df = load_design_results("capex_tot", results_path, year, components_df)

# Load size data
components_df = load_design_results("size", results_path, year, components_df)

print(components_df)

     node        component     capex_tot        size
0    BEL1              ASU  0.000000e+00    0.000000
1    BEL1        Boiler_El  8.627640e+05   59.919149
2    BEL1   CrackerFurnace  1.775715e+08  723.414121
3    BEL1     Electrolyzer  0.000000e+00    0.000000
4    BEL1     HBfeed_mixer  0.000000e+00    0.000000
..    ...              ...           ...         ...
103   NL3       WGS_syngas  0.000000e+00    0.000000
104   NL3  eCrackerFurnace  0.000000e+00    0.000000
105   NL3      eSMR_syngas  0.000000e+00    0.000000
106   NL3    feedgas_mixer  0.000000e+00    0.000000
107   NL3             rWGS  0.000000e+00    0.000000

[108 rows x 4 columns]


In [3]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import h5py

def _to_scalar(x):
    if isinstance(x, (np.generic,)):
        return x.item()
    x = np.asarray(x)
    if x.shape == ():
        return x.item()
    if x.size == 1:
        return x.reshape(-1)[0].item()
    return x

def split_edge_id(edge_id: str, node_names: list[str]) -> tuple[str, str]:
    """
    EDGE_ID nel file sembra essere la concatenazione dei due nodi (es. BEL1NL3, DENL2, BEL2DE).
    Per separarla in (from, to), cerco un nodo che sia prefisso e uno che sia suffisso.
    """
    # prova tutti i prefissi possibili
    for n_from in sorted(node_names, key=len, reverse=True):
        if edge_id.startswith(n_from):
            rest = edge_id[len(n_from):]
            # il resto deve essere un nodo valido
            if rest in node_names:
                return n_from, rest
    raise ValueError(f"Impossibile splittare edge_id='{edge_id}' con node_names={node_names}")

def list_datasets(group: h5py.Group) -> list[str]:
    out = []
    def _rec(g, prefix=""):
        for k, v in g.items():
            if isinstance(v, h5py.Dataset):
                out.append(prefix + k)
            elif isinstance(v, h5py.Group):
                _rec(v, prefix + k + "/")
    _rec(group, "")
    return out

def read_network_edges(
    h5_path: str | Path,
    year: str = "2022",
    network: str = "hydrogenPipelineOnshore",
    datasets: str = "capex_tot"
) -> pd.DataFrame:
    """
    Read edges of a network from the HDF5 file and extract specified datasets for each edge.
    Assumes edge_id is the name of the group under design/networks/{year}/{network}/, and datasets are under design/networks/{year}/{network}/{edge_id}/{dataset}.
    """
    h5_path = Path(h5_path)
    base = f"design/networks/{year}/{network}"

    with h5py.File(h5_path, "r") as f:
        if base not in f:
            raise KeyError(f"Path '{base}' not found in the file.")
        g = f[base]
        edge_ids = list(g.keys())

        rows = []
        for eid in edge_ids:
            row = {"edge_id": eid}
            p = f"{base}/{eid}/{datasets}"
            row[datasets] = _to_scalar(f[p][()]) if p in f else np.nan
            rows.append(row)

    return pd.DataFrame(rows)

def attach_geometry(edges_df: pd.DataFrame, nodes_csv: str | Path) -> pd.DataFrame:
    nodes = pd.read_csv(nodes_csv)
    # nel tuo csv: cluster, latitude, longitude
    node_names = nodes["cluster"].astype(str).tolist()
    coords = nodes.set_index("cluster")[["latitude", "longitude"]]

    fr_to = edges_df["edge_id"].apply(lambda s: split_edge_id(str(s), node_names))
    edges_df = edges_df.copy()
    edges_df["from"] = fr_to.apply(lambda x: x[0])
    edges_df["to"]   = fr_to.apply(lambda x: x[1])

    edges_df["from_lat"] = edges_df["from"].map(coords["latitude"])
    edges_df["from_lon"] = edges_df["from"].map(coords["longitude"])
    edges_df["to_lat"]   = edges_df["to"].map(coords["latitude"])
    edges_df["to_lon"]   = edges_df["to"].map(coords["longitude"])

    return edges_df

def make_folium_map(edges_df: pd.DataFrame, nodes_csv: str | Path, size_col: str = "size"):
    import folium

    nodes = pd.read_csv(nodes_csv)
    center = [nodes["latitude"].mean(), nodes["longitude"].mean()]
    m = folium.Map(location=center, zoom_start=6)

    # scala spessore linee
    s = edges_df[size_col].astype(float)
    s_max = np.nanmax(s.values) if np.isfinite(s).any() else 1.0

    # marker nodi
    for _, r in nodes.iterrows():
        folium.CircleMarker(
            location=[r["latitude"], r["longitude"]],
            radius=5,
            popup=str(r["cluster"]),
            fill=True
        ).add_to(m)

    # linee archi
    for _, r in edges_df.iterrows():
        if pd.isna(r["from_lat"]) or pd.isna(r["to_lat"]):
            continue
        w = 1 + 8 * (float(r[size_col]) / s_max) if pd.notna(r[size_col]) and s_max > 0 else 2
        folium.PolyLine(
            locations=[[r["from_lat"], r["from_lon"]], [r["to_lat"], r["to_lon"]]],
            weight=w,
            opacity=0.8,
            popup=f"{r['from']}→{r['to']} | {size_col}={r[size_col]:.3g} | capex={r.get('capex', np.nan):.3g}"
        ).add_to(m)

    return m


h5_path = r"output\20260206153536-1\optimization_results.h5"
nodes_csv = r"plants_data\plants_clusters_summary.csv"
year = "2022"

edges = read_network_edges(h5_path, year=year, network="hydrogenPipelineOnshore", datasets="capex")
edges = attach_geometry(edges, nodes_csv)

m = make_folium_map(edges, nodes_csv, size_col="capex")
m  # in notebook/Jupyter te la renderizza
